In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
from torchtext.vocab import build_vocab_from_iterator
import tqdm
import evaluate

D:\ANACONDA\envs\nlp_xryh\lib\site-packages\torchtext\vocab\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
D:\ANACONDA\envs\nlp_xryh\lib\site-packages\torchtext\utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)


In [2]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True


### 设置 HF_ENDPOINT='https://hf-mirror.com' 会将 Hugging Face 库默认请求的 API 地址从官方源 https://huggingface.co 替换为国内镜像站 hf-mirror.com。
### 使用镜像站是为了在国内网络环境下避免连接不稳定、速度慢或被阻断的问题，从而稳定快速地下载数据集或模型。
### load_dataset("bentrevett/multi30k") 会自动检查本地缓存，若不存在则从 Hugging Face 仓库下载、解压并解析该数据集，返回可用于训练的 Dataset 或 DatasetDict 对象

In [3]:
from datasets import Dataset, DatasetDict

# 定义文件路径（请改成你的实际路径）
train_jsonl = r"D:\LIST(3)\train.jsonl"
val_jsonl = r"D:\LIST(3)\val.jsonl"
test_jsonl = r"D:\LIST(3)\test.jsonl"

# 从 JSONL 文件加载 Dataset 对象
train_dataset = Dataset.from_json(train_jsonl)
val_dataset = Dataset.from_json(val_jsonl)
test_dataset = Dataset.from_json(test_jsonl)

# 组合成 DatasetDict（与 load_dataset 返回的格式一致）
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

# 验证
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})
{'en': 'Two young, White males are outside near many bushes.', 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}


### 数据集对象（一个DatasetDict）包含训练、验证和测试集，每个集合中的样本数量，以及每个集合中的特征（“en”和“de”）。

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

In [5]:
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

### 可以索引每个数据集来查看单个示例。每个例子都有两个特征：“en”和“de”，是对应的英语和德语。

In [6]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

### 分词器 Tokenizers

In [7]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

### 可以使用.tokenizer方法调用每个spaCy模型的分词器，该方法接受字符串并返回Token对象序列。我们可以使用text属性从Token对象中获取字符串。

In [8]:
string = "What a lovely day it is today!"

[token.text for token in en_nlp.tokenizer(string)]

['What', 'a', 'lovely', 'day', 'it', 'is', 'today', '!']

### 这个函数的作用是将一个包含英文和德文句子的样本（example）进行分词、截断、可选小写化，并在首尾添加起始符（sos_token）和结束符（eos_token），最终返回处理后的 token 列表

In [9]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

####  <sos>和 <eos> 的含义：<sos>（start of sentence）表示句子开头，<eos>（end of sentence）表示句子结尾，它们作为特殊标记加入 token 序列，用于帮助模型学习何时开始生成或停止生成。
#### map 函数的作用：将 tokenize_example 函数应用到数据集的每一个样本上，对每条“en”和“de”文本进行分词、截断、小写化并添加首尾标记，生成新的 token 字段，实现数据集的批量预处理。

In [10]:
max_length = 1_000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

### 重新打印train_data[0]，验证小写字符串列表以及序列标记的开始/结束符已被成功添加

In [11]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

### 词汇表 Vocabularies

In [15]:
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

en_vocab = build_vocab_from_iterator(
    train_data["en_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

de_vocab = build_vocab_from_iterator(
    train_data["de_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

### 验证词汇表，分别打印英语词汇表和德语词汇表的前十个Token。

In [16]:
en_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', 'a', '.', 'in', 'the', 'on', 'man']

In [17]:
de_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', '.', 'ein', 'einem', 'in', 'eine', ',']

### 使用get_stoi（stoi = "string to int "）方法获取指定的Token的索引。

In [18]:
en_vocab["the"]

7

In [19]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

In [20]:
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

### 观察从Token列表到索引列表的转换。

In [21]:
tokens = ["i", "love", "watching", "crime", "shows"]
en_vocab.lookup_indices(tokens)

[956, 2169, 173, 0, 821]

### 观察从索引列表到Token列表的转换。

In [22]:
en_vocab.lookup_tokens(en_vocab.lookup_indices(tokens))

['i', 'love', 'watching', '<unk>', 'shows']

#### "crime" 被转换成 <unk> 的原因是：词汇表（en_vocab 或 de_vocab）通常只基于训练集构建，并且会过滤掉低频词（例如通过 min_freq=2 只保留出现次数 ≥2 的词）。如果 "crime" 在训练集中未出现或出现次数低于设定阈值，它就不会被加入词汇表。当验证集或测试集中包含这个词时，lookup_indices 方法无法找到对应的索引，因此自动返回 <unk> 的索引。这是处理未登录词（OOV）的标准机制。

#### 将 token 序列转换为整数 ID 序列

In [23]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

In [24]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

### 重新打印train_data[0]，验证"en_ids" and "de_ids"被成功添加。

In [25]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>'],
 'en_ids': [2, 16, 24, 15, 25, 778, 17, 57, 80, 202, 1312, 5, 3],
 'de_ids': [2, 18, 26, 253, 30, 84, 20, 88, 7, 15, 110, 7647, 3171, 4, 3]}

In [26]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

### 重新打印train_data[0]，验证“en_ids”和“de_ids”特征被转换为了张量。

In [27]:
train_data[0]

{'en_ids': tensor([   2,   16,   24,   15,   25,  778,   17,   57,   80,  202, 1312,    5,
            3]),
 'de_ids': tensor([   2,   18,   26,  253,   30,   84,   20,   88,    7,   15,  110, 7647,
         3171,    4,    3]),
 'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

#### 为数据加载提供批次处理和对齐功能

In [28]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [29]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [30]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

### 编码器 Encoder

输入参数
input_dim：源语言词汇表的大小（即输入序列中 token 索引的最大值 + 1），用于定义词嵌入层的输入维度。
embedding_dim：词嵌入向量的维度，每个 token 会被映射为一个 embedding_dim 维的稠密向量。
hidden_dim：LSTM 隐藏状态的维度，也是每个时间步输出的特征维度。
n_layers：LSTM 的层数（堆叠层数），层数越多模型容量越大。
dropout：Dropout 的比例，用于在训练时随机丢弃部分神经元以防止过拟合（仅在多层 LSTM 的层间生效）。
核心组件
词嵌入层（self.embedding）：
nn.Embedding(input_dim, embedding_dim)
将输入的整数索引序列转换为稠密的嵌入向量序列。
LSTM 层（self.rnn）：
nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
接收嵌入向量序列，输出隐藏状态序列和最后时间步的隐藏状态/细胞状态。
设置 dropout 参数后，除最后一层外，各层之间会应用 Dropout。
Dropout 层（self.dropout）：
nn.Dropout(dropout)
在词嵌入层之后对嵌入向量进行 Dropout 正则化。
forward 处理流程
输入 src：形状为 [src length, batch size]，即源语言序列（按时间步排列，每个元素是 token 索引）。
嵌入与 Dropout：
embedded = self.dropout(self.embedding(src))
先将索引映射为嵌入向量（形状 [src length, batch size, embedding_dim]），再应用 Dropout。
送入 LSTM：
outputs, (hidden, cell) = self.rnn(embedded)
LSTM 处理整个输入序列，返回：
outputs：所有时间步的顶层隐藏状态（形状 [src length, batch size, hidden_dim * num_directions]，由于是单向 LSTM，num_directions=1，所以最后两维为 [batch size, hidden_dim]）。
hidden：最后一层的最终隐藏状态（形状 [n_layers, batch size, hidden_dim]）。
cell：最后一层的最终细胞状态（形状同 hidden）。
输出
函数返回 (hidden, cell)，不返回 outputs。
hidden 和 cell 将被传递给解码器（Decoder）作为初始状态，实现编码器-解码器架构中的信息传递。

In [31]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src length, batch size]
        embedded = self.dropout(self.embedding(src))
        # embedded = [src length, batch size, embedding dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs = [src length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # outputs are always from the top hidden layer
        return hidden, cell

### 解码器 Decoder

Decoder 类实现了一个单步解码器，即在每个时间步接收上一时刻生成的 token（或训练时的真实 token），结合编码器传来的隐藏状态 hidden 和细胞状态 cell，输出当前时刻对目标语言词汇的预测概率分布，并更新 RNN 的状态。

In [32]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # n directions in the decoder will both always be 1, therefore:
        # hidden = [n layers, batch size, hidden dim]
        # context = [n layers, batch size, hidden dim]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, embedding dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output = [seq length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # seq length and n directions will always be 1 in this decoder, therefore:
        # output = [1, batch size, hidden dim]
        # hidden = [n layers, batch size, hidden dim]
        # cell = [n layers, batch size, hidden dim]
        prediction = self.fc_out(output.squeeze(0))
        # prediction = [batch size, output dim]
        return prediction, hidden, cell

### Seq2Seq

Seq2Seq 类将编码器（Encoder）和解码器（Decoder）组合成一个完整的序列到序列模型，并实现了带 Teacher Forcing 的训练逻辑。
初始化 __init__
接收编码器、解码器和设备（CPU/GPU）作为参数。
断言编码器和解码器的 hidden_dim 和 n_layers 必须相等，因为解码器需要以编码器的最终隐藏状态作为初始状态。
forward 函数流程
输入：
src：源语言序列，形状 [src_len, batch_size]
trg：目标语言序列，形状 [trg_len, batch_size]（包含 <sos> 和 <eos>）
teacher_forcing_ratio：使用 Teacher Forcing 的概率（0~1）
处理步骤：
初始化输出张量：
outputs = torch.zeros(trg_len, batch_size, trg_vocab_size)
用于存储解码器每个时间步的预测结果（logits）。
编码：
hidden, cell = self.encoder(src)
将源序列输入编码器，获得最终的隐藏状态和细胞状态，作为解码器的初始状态。
解码器初始输入：
input = trg[0, :]
取目标序列的第一个 token（通常是 <sos>），作为解码器第一个时间步的输入。
循环解码（for t in range(1, trg_len)）：
前向：output, hidden, cell = self.decoder(input, hidden, cell)
解码器根据当前输入和状态，输出当前时间步的预测 logits（形状 [batch_size, output_dim]）和更新后的状态。
存储预测：outputs[t] = output
将当前预测存入输出张量的对应位置（注意 t=0 位置保留为 0，因为不预测第一个 token）。
Teacher Forcing 决策：
teacher_force = random.random() < teacher_forcing_ratio
确定下一个输入：
top1 = output.argmax(1)（当前预测概率最高的 token）
input = trg[t] if teacher_force else top1
若使用 Teacher Forcing，则下一个输入为真实目标序列的下一个 token trg[t]；否则使用模型自己预测的 token top1。
继续循环。
返回：outputs（形状 [trg_len, batch_size, output_dim]），包含每个时间步对目标词汇的预测 logits（outputs[0] 为全零）。
Teacher Forcing 机制
定义：在解码器训练过程中，以一定概率使用真实目标 token（而不是模型上一时刻的预测 token）作为下一个时间步的输入。
作用：加速模型收敛、缓解误差累积（防止早期预测错误导致后续序列完全偏离）。
实现：通过 teacher_forcing_ratio 控制概率，随机决定每个时间步是否使用真实标签。推理时（不使用 Teacher Forcing）则完全依赖模型自己的预测。
示例：ratio=0.75 表示 75% 的时间步输入真实 token，25% 的时间步输入模型预测的 token。

In [33]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        # teacher_forcing_ratio is probability to use teacher forcing
        # e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        # last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # first input to the decoder is the <sos> tokens
        input = trg[0, :]
        # input = [batch size]
        for t in range(1, trg_length):
            # insert input token embedding, previous hidden and previous cell states
            # receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, hidden dim]
            # cell = [n layers, batch size, hidden dim]
            # place predictions in a tensor holding predictions for each token
            outputs[t] = output
            # decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio
            # get the highest predicted token from our predictions
            top1 = output.argmax(1)
            # if teacher forcing, use actual next token as next input
            # if not, use predicted token
            input = trg[t] if teacher_force else top1
            # input = [batch size]
        return outputs

### 模型训练

In [34]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 编码器初始化
encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

# 解码器初始化
decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

# Seq2Seq模型整合
model = Seq2Seq(encoder, decoder, device).to(device)

# 权重初始化
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

# 优化器 optimizer
optimizer = optim.Adam(model.parameters())

# 损失函数 Loss Function
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

The model has 13,898,501 trainable parameters


### Training Loop:

In [35]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    # 定义训练函数，接收模型、数据加载器、优化器、损失函数、梯度裁剪阈值、teacher forcing概率、设备
    model.train()
    # 将模型设置为训练模式（启用dropout等训练专用层）
    epoch_loss = 0
    # 初始化整个epoch的总损失为0
    for i, batch in enumerate(data_loader):
        # 遍历数据加载器，i是批次索引，batch是一个批次的数据
        src = batch["de_ids"].to(device)
        # 从batch中取出源语言（德语）的整数序列，并移动到指定设备（CPU/GPU）
        trg = batch["en_ids"].to(device)
        # 从batch中取出目标语言（英语）的整数序列，并移动到指定设备
        # src = [src length, batch size]
        # 注释说明src的形状：序列长度 × 批次大小
        # trg = [trg length, batch size]
        # 注释说明trg的形状：序列长度 × 批次大小
        optimizer.zero_grad()
        # 清空优化器中的梯度（因为梯度会累积）
        output = model(src, trg, teacher_forcing_ratio)
        # 将源序列、目标序列和teacher forcing概率传入模型进行前向传播，得到输出张量
        # output = [trg length, batch size, trg vocab size]
        # 注释说明output的形状：目标序列长度 × 批次大小 × 目标词汇表大小
        output_dim = output.shape[-1]
        # 获取目标词汇表大小（output的最后一个维度）
        output = output[1:].view(-1, output_dim)
        # 去掉output的第一个时间步（对应<sos>的预测，通常不使用），然后重塑为二维张量：
        # 行数 = (目标序列长度-1) × 批次大小，列数 = 目标词汇表大小
        # output = [(trg length - 1) * batch size, trg vocab size]
        # 注释说明重塑后的形状
        trg = trg[1:].view(-1)
        # 去掉trg的第一个token（<sos>），然后展平为一维张量，长度 = (目标序列长度-1) × 批次大小
        # trg = [(trg length - 1) * batch size]
        # 注释说明展平后的形状
        loss = criterion(output, trg)
        # 计算损失：比较模型预测（output）与真实目标序列（trg）
        loss.backward()
        # 反向传播，计算梯度
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        # 梯度裁剪，防止梯度爆炸，将梯度范数限制在clip值以内
        optimizer.step()
        # 更新模型参数
        epoch_loss += loss.item()
        # 累加当前批次的损失值（loss.item()取出标量）
    return epoch_loss / len(data_loader)
    # 返回平均每个批次的损失（epoch总损失除以批次数）

### Evaluation Loop:

In [5]:

def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["de_ids"].to(device)
            trg = batch["en_ids"].to(device)
            # src = [src length, batch size]
            # trg = [trg length, batch size]
            output = model(src, trg, 0)  # turn off teacher forcing
            # output = [trg length, batch size, trg vocab size]
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            # output = [(trg length - 1) * batch size, trg vocab size]
            trg = trg[1:].view(-1)
            # trg = [(trg length - 1) * batch size]
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

### 模型训练

In [6]:
n_epochs = 1 # 因模型训练对计算资源要求较高，此处只设立了一轮训练。
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

  0%|                                                                                            | 0/1 [00:00<?, ?it/s]


NameError: name 'train_fn' is not defined

### 模型验证

In [ ]:
model.load_state_dict(torch.load("tut1-model.pt"))

In [ ]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        if lower:
            tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [ ]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

In [ ]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

### 运行下方单元格，得到测试集第0个索引的翻译

In [ ]:
translation